In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l4.1 Residual stream

A transformer block does not replace its input, it *adds* to it: `x + f(x)`.
That running sum is the **residual stream**. Gradients flow straight through
the `+`, so even deep stacks keep training instead of vanishing.

In [ ]:
from torch import nn

torch.manual_seed(0)
dim = 8
x = torch.randn(1, 4, dim)
sublayer = nn.Linear(dim, dim)
delta = sublayer(x)
residual_out = x + delta   # the residual connection
print("output = input + a learned delta")

Each block reads the stream, computes a delta, and writes it back. Information
from layer 1 can survive untouched all the way to the output.

In [ ]:
# (x + delta) - x recovers delta up to floating-point cancellation.
assert torch.allclose(residual_out - x, delta, atol=1e-5)